In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from collections import defaultdict
from data_preprocess import encode_features, PCA, FeatureSelection

In [2]:
class NaiveBayesClassifier:
    def __init__(self, data=None):
        self.edidble_counts = 0
        self.poisonous_counts = 0
        self.edidble_prob = 0.5
        self.poisonous_prob = 0.5
        self.data = data
        self.edidble_features_likelihoods = {}
        self.poisonous_features_likelihoods = {}
        
    def loadData(self, data):
        self.data = data

    def compute_initial_probs(self):
        data = self.data
        nrecords = len(data)
        
        counts = data['class'].value_counts()
        edidble_counts = counts['e']
        poisonous_counts = counts['p']
        
        total = edidble_counts + poisonous_counts
        
        self.edidble_counts = edidble_counts
        self.poisonous_counts = poisonous_counts
        
        self.edidble_prob = edidble_counts / total
        self.poisonous_prob = poisonous_counts / total
    
        
        return self.edidble_prob, self.poisonous_prob
    
    def compute_histograms(self):
        df = self.data
        
        edidble_features_freq = defaultdict(dict)
        poisonous_features_freq = defaultdict(dict)
        total_edible_observations = self.edidble_counts * 22
        total_poisonous_observations = self.poisonous_counts * 22
        
        for _, row in df.iterrows():
            label = row['class']
            
            for feature in df.columns:
                if feature == 'class': continue
                
                value = row[feature]
                if label == 'e':
                    edidble_features_freq[feature][value] = edidble_features_freq[feature].get(value, 0) + 1
                else:
                    poisonous_features_freq[feature][value] = poisonous_features_freq[feature].get(value, 0) + 1
                    
        return edidble_features_freq, poisonous_features_freq, total_edible_observations, total_poisonous_observations
    

    def compute_likelihoods(self): 
        edidble_features_freq, poisonous_features_freq, total_edible_observations, total_poisonous_observations = self.compute_histograms()
        
        edidble_features_likelihoods = defaultdict(dict)
        poisonous_features_likelihoods = defaultdict(dict)
        
        for feature in edidble_features_freq:
            for value, freq in edidble_features_freq[feature].items():
                edidble_features_likelihoods[feature][value] = freq / total_edible_observations
        
        for feature in poisonous_features_freq:
            for value, freq in poisonous_features_freq[feature].items():
                poisonous_features_likelihoods[feature][value] = freq / total_poisonous_observations
                
        self.edidble_features_likelihoods, self.poisonous_features_likelihoods = edidble_features_likelihoods, poisonous_features_likelihoods
                    
        return self.edidble_features_likelihoods, self.poisonous_features_likelihoods
    
    def fit(self):
        self.compute_initial_probs()
        return self.compute_likelihoods()
    
    def predict(self, row: pd.Series):
        edidble_pred_score = np.log(self.edidble_prob)
        poisonous_pred_score = np.log(self.poisonous_prob)
        
        edidble_features_likelihoods = self.edidble_features_likelihoods
        poisonous_features_likelihoods = self.poisonous_features_likelihoods
        
        for feature, value in row.items():
            if feature == 'class': continue
            edidble_pred_score += np.log(edidble_features_likelihoods[feature].get(value, 1e-10))
            poisonous_pred_score += np.log(poisonous_features_likelihoods[feature].get(value, 1e-10))
            

        if edidble_pred_score > poisonous_pred_score:
            return 'e'
        else:
            return 'p'
        
        
    def test(self, test_data: pd.DataFrame):
        for _, row in test_data.iterrows():
            label = row['class']
            pred = self.predict(row=row)
            print(f'{_}: {"correct" if label == pred else "wrong"}')
            

In [3]:
df = pd.read_csv('data/mushrooms.csv')
train_df, test_df = train_test_split(df, test_size=0.2)
nb = NaiveBayesClassifier(data=train_df)
nb.fit()
nb.test(test_data=test_df)

953: correct
7597: correct
3786: correct
3137: correct
7896: correct
3554: correct
6648: correct
100: correct
1640: correct
818: correct
8039: correct
691: correct
2612: correct
3353: correct
2315: correct
7658: correct
7737: correct
1066: correct
656: correct
626: correct
5172: correct
2176: correct
6118: correct
7596: correct
5054: correct
6248: correct
7250: correct
4681: correct
2745: correct
3029: correct
5628: correct
177: correct
8086: correct
756: correct
2224: correct
60: correct
2867: correct
1827: correct
2636: correct
1580: correct
35: correct
2989: correct
3870: correct
5051: correct
7796: correct
6828: correct
1285: correct
5504: correct
1117: correct
7181: correct
1021: correct
1405: correct
7465: correct
7833: correct
4732: correct
7963: correct
4149: correct
6994: correct
3457: correct
6659: correct
1960: correct
6913: correct
6700: correct
6325: correct
5061: correct
2177: correct
6139: correct
5690: correct
2463: correct
6036: correct
4091: correct
6905: correct
3922

In [4]:
encoded_df = encode_features(df)
pca = PCA(encoded_df)
print(pca.compute_fprime())

[[ 0.22820662 -1.93656097 -1.65417317 ... -1.86050705  7.29306843
  -4.83553281]
 [-0.34547209  4.79691211  2.46436226 ...  0.47024368 -1.25169255
  -3.58375805]
 [ 1.42442514  3.51166784  3.88082715 ... -1.19021189  1.24197629
   0.11250607]]
